In [1]:
# conda env 'vis2'
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent.parent.parent

import sys
import os
sys.path.append(str(PROJECT_ROOT))

from src.utils.util import load_config
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import glob
import at3d

from collections import defaultdict

In [2]:
def parse_at3d_output(dir):
    loss_file = dir / 'loss.csv'
    log_file = dir / 'at3d.log'
    info = {}
    try:
        with open(log_file, 'r') as file:
            lines = list(file)
            info['termination'] = lines[-1].strip()
            info['F'] = float(lines[-3].split()[2])
            info['N'] = int(lines[-4].split()[0])
            info['Tit'] = int(lines[-4].split()[1])
            info['Tnf'] = int(lines[-4].split()[2])
            info['Tnint'] =  int(lines[-4].split()[3])
            info['Skip'] = int(lines[-4].split()[4])
            info['Nact'] = int(lines[-4].split()[5])
            info['Projg'] = float(lines[-4].split()[6].replace('D','E'))
            info['max_cell_splitting'] = False
            for l in lines:
                if('Further cell splitting' in l):
                    info['max_cell_splitting'] = True
                if('Execution time' in l):
                    info['run time'] = float(l.split()[2])
    except FileNotFoundError:
        print('Error: File not found at {}'.format(log_file))
    except Exception as e:
        print(f"Error while parsing log file: {e}")

    try:
        with open(loss_file, 'r') as file:
            lines = list(file)
            info['total_obj_fn_time'] = float(lines[-1].split()[2])
    except FileNotFoundError:
        print('Error: File not found at {}'.format(loss_file))
    except Exception as e:
        print(f"Error while parsing log file: {e}")
    
    return info

In [3]:
def load_retreival(dir):
    iterations = dir / 'iterations'
    assert iterations.is_dir()
    files = sorted(iterations.glob('iter*.nc'))
    sensors, solvers, rte_grid = at3d.util.load_forward_model(files[-1])
    data = {
        'files': files,
        'sensors': sensors,
        'solvers': solvers,
        'rte_grid': rte_grid,
        'log': parse_at3d_output(dir),
    }
    return data

In [4]:
def traverse(dir, data, parent_key=()):
    name = dir.name
    parent_key += (name,)
    iterations = dir / 'iterations'
    if(iterations.is_dir()):
        data[parent_key] = load_retreival(dir)
    else:
        for n in dir.iterdir():
            traverse(dir / n, data, parent_key)


In [5]:
V = 'V08'
VERSION = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / V
DEFAULT = VERSION / 'default'

config = load_config(DEFAULT / 'config.yml')

In [6]:
data = defaultdict(defaultdict)
traverse(VERSION, data)

/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '71'. Max cell optical depth: '3.6754472255706787'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "
/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3

In [7]:
def print_nested_dictionary_keys(nested_dict, indent=0):
    """
    Recursively prints keys from a nested dictionary with indentation.

    Args:
        nested_dict (dict): The nested dictionary to print keys from.
        indent (int): The current indentation level.
    """
    for key, value in nested_dict.items():
        print('  ' * indent + str(key)) # Print key with indentation
        if isinstance(value, dict):
            print_nested_dictionary_keys(value, indent + 1) # Recursive call for nested dictionaries

print_nested_dictionary_keys(data)

('V08', 'CVM', 'LCL_CTH')
  files
  sensors
    MISR
      sensor_list
      uncertainty_model
  solvers
    0.67
  rte_grid
  log
    termination
    F
    N
    Tit
    Tnf
    Tnint
    Skip
    Nact
    Projg
    max_cell_splitting
    run time
    total_obj_fn_time
('V08', 'CVM', 'unconstrained')
  files
  sensors
    MISR
      sensor_list
      uncertainty_model
  solvers
    0.67
  rte_grid
  log
    termination
    F
    N
    Tit
    Tnf
    Tnint
    Skip
    Nact
    Projg
    max_cell_splitting
    run time
    total_obj_fn_time
('V08', 'CVM', 'unconstrained_darksurf')
  files
  sensors
    MISR
      sensor_list
      uncertainty_model
  solvers
    0.67
  rte_grid
  log
    termination
    F
    N
    Tit
    Tnf
    Tnint
    Skip
    Nact
    Projg
    max_cell_splitting
    run time
    total_obj_fn_time
('V08', 'surf_wind', '1p33')
  files
  sensors
    MISR
      sensor_list
      uncertainty_model
  solvers
    0.67
  rte_grid
  log
    termination
    F
    N
    

### Error Metrics

$\bm{\beta_p}$ is the perturbed extinction coefficient vector. $\bm{\beta_r}$ is the reference extinction coefficient vector.

Relative Bias (Jesse's relative bias):

$\text{Relative Bias} = 100 \times \frac{||\bm{\beta_p}||_1 - ||\bm{\beta_r}||_1}{||\bm{\beta_r}||_1}$

Relative Difference (Jesse's relative RMSE):

$\text{Relative Difference} = 100 \times \frac{||\bm{\beta_p} - \bm{\beta_r}||_2}{||\bm{\beta_r}||_2}$

$||\bm{x}||_1 = \sum_i |x_i|$ and $||\bm{x}||_2 = \left(\sum_i x_i^2\right)^{\frac{1}{2}}$

Mean Absolute Error:

$\text{MAE} = \frac{1}{n} \sum_{i=1}^n |\beta_{p,i} - \beta_{r, i}|$

Root Mean Square Error:

$\text{RMSE} = \left(\frac{1}{n} \sum_{i=1}^n \left(\beta_{p,i} - \beta_{r,i}\right)^2\right)^\frac{1}{2} = \left(\frac{1}{n} ||\bm{\beta_p} - \bm{\beta_r}||_2^2\right)^{\frac{1}{2}}$

Correlation Coefficient (Pearson's r):

$r = \frac{\sum_i (\beta_{p,i} - \bar{\bm{\beta_p}})(\beta_{r,i} - \bar{\bm{\beta_r}})}{\sqrt{\sum_i (\beta_{p,i} - \bar{\bm{\beta_p}})^2} \sqrt{\sum_i (\beta_{r,i} - \bar{\bm{\beta_r}})^2}}$

Normalized Mean Bias (NMB):

$\text{NMB} = \frac{\sum_i (\beta_{p,i} - \beta_{r,i})}{\sum_i \beta_{r,i}}$


In [8]:
def calc_error_metrics(reference, data):
    if('solvers' in list(data.keys())):
        ext_tmp = np.ravel(data['solvers'][0.67].medium['cloud'].extinction.data)
        ext_ref = np.ravel(reference['solvers'][0.67].medium['cloud'].extinction.data)
        data['relative difference'] = 100 * (np.sum((ext_tmp - ext_ref)**2))**0.5 / (np.sum(ext_ref)**2)**0.5
        data['relative bias'] = 100 * (np.sum(np.abs(ext_tmp)) - np.sum(np.abs(ext_ref))) / (np.sum(np.abs(ext_ref)))
        data['MAE'] = np.sum(np.abs(ext_tmp - ext_ref)) / len(ext_tmp)
        data['RMSE'] = (np.sum((ext_tmp - ext_ref)**2) / len(ext_tmp))**0.5
        data['r'] = (np.sum((ext_tmp - np.mean(ext_tmp)) * (ext_ref - np.mean(ext_ref)))) / ((np.sum((ext_tmp - np.mean(ext_tmp))**2)**0.5) * (np.sum((ext_ref - np.mean(ext_ref))**2)**0.5))
        data['NMB'] = np.sum(ext_tmp - ext_ref) / np.sum(ext_ref)

        tmp = data['solvers'][0.67].medium['cloud'].copy(deep=True)
        mask = (tmp.extinction.data > 0).astype(np.int64)
        tmp['cvm'] = (['x', 'y', 'z'], mask)
        cv = tmp.cvm.integrate(['x','y','z']).data
        tmp_ref = reference['solvers'][0.67].medium['cloud'].copy(deep=True)
        mask = (tmp_ref.extinction.data > 0).astype(np.int64)
        tmp_ref['cvm'] = (['x', 'y', 'z'], mask)
        cv_ref = tmp_ref.cvm.integrate(['x','y','z']).data
        data['cloud volume'] = cv
        data['delta volume'] = cv - cv_ref

        tmp['tau'] = tmp.extinction.integrate(['z'])
        tmp['cloud_mask'] = (tmp.tau > 0).astype(np.int64)
        tmp['cf'] = tmp.cloud_mask.sum() / (tmp.sizes['x'] * tmp.sizes['y'])
        data['cloud fraction'] = tmp.cf.data
        data['tau bar'] = tmp.tau.where(tmp.cloud_mask > 0).mean().data
    else:
        for n in data:
            calc_error_metrics(reference, data[n])

In [9]:
reference = data[(V, 'default')]
calc_error_metrics(reference, data)

In [10]:
cvm = xr.load_dataset(config['cloud_volume_mask_file'])
vol_cvm = cvm.cloud_volume_mask.integrate(['x','y','z']).data
print('Default CVM Volume: {:.3f} km**3'.format(vol_cvm))

Default CVM Volume: 88.164 km**3


In [11]:
print('{:<53} | {:<8} | {:<8} | {:<6} | {:<7} | {:<9} '.format('Path', 'Bias (%)', 'Diff (%)', 'CF', 'tau bar', 'V (km**3)'))
print('-'*100)
for k, v in data.items():
    print_line = True
    for i in k:
        if('obsolete' in i):
            print_line = False
    if(print_line):
        print('{:<53} | {:<8.2f} | {:<8.2f} | {:<6.3f} | {:<7.3f} | {:<9.2f}'.format(str(k), v['relative bias'], v['relative difference'], v['cloud fraction'], v['tau bar'], v['cloud volume']))

Path                                                  | Bias (%) | Diff (%) | CF     | tau bar | V (km**3) 
----------------------------------------------------------------------------------------------------
('V08', 'CVM', 'LCL_CTH')                             | 0.03     | 0.31     | 0.266  | 1.347   | 18.29    
('V08', 'CVM', 'unconstrained')                       | 25.44    | 2.98     | 0.357  | 1.124   | 30.64    
('V08', 'CVM', 'unconstrained_darksurf')              | 54.09    | 3.11     | 0.437  | 1.136   | 43.58    
('V08', 'surf_wind', '1p33')                          | 20.74    | 1.49     | 0.353  | 1.226   | 22.26    
('V08', 'surf_wind', '0p1')                           | -54.38   | 1.77     | 0.118  | 1.380   | 8.42     
('V08', 'cloud_motion', 'fast_unconstrained')         | 21.43    | 3.18     | 0.380  | 1.021   | 32.83    
('V08', 'cloud_motion', 'slow_unconstrained')         | 37.63    | 3.96     | 0.354  | 1.200   | 29.10    
('V08', 'cloud_motion', 'slow')           

In [12]:
print('{:<53} | {:<5} | {:<4} | {:<6} | {:<6} | {:<7} | {:<6} | {:<14} | {:<20} '.format(
    'Path',
    'N',
    'Tit',
    't (s)',
    's/iter',
    '% SHDOM',
    'F',
    'Max Cell Split',
    'Termination'
))
print('-'*170)
for k, v in data.items():
    if('Tit' in v['log']):
        print('{:<53} | {:<5d} | {:<4d} | {:<6.0f} | {:<6.1f} | {:<7.2f} | {:<6.2f} | {:<14} | {:<20} '.format(
            str(k),
            v['log']['N'],
            v['log']['Tit'],
            v['log']['run time'],
            v['log']['run time']/v['log']['Tit'],
            100 - 100*v['log']['total_obj_fn_time']/v['log']['run time'],
            v['log']['F'],
            v['log']['max_cell_splitting'],
            v['log']['termination']
        ))

Path                                                  | N     | Tit  | t (s)  | s/iter | % SHDOM | F      | Max Cell Split | Termination          
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
('V08', 'CVM', 'LCL_CTH')                             | 13162 | 20   | 5158   | 257.9  | 0.48    | 560.33 | 0              | CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH 
('V08', 'CVM', 'unconstrained')                       | 59535 | 35   | 26181  | 748.0  | 0.29    | 547.39 | 0              | CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH 
('V08', 'CVM', 'unconstrained_darksurf')              | 59535 | 24   | 13673  | 569.7  | 0.37    | 527.85 | 1              | CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH 
('V08', 'surf_wind', '1p33')                          | 11658 | 47   | 17320  | 368.5  | 0.52    | 538.10 | 0              | CONVERGENCE: REL_REDUCTION_OF_F_<=_